# ONE TIME RUNS

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJ = '/content/drive/MyDrive/TAC_459/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch

!pip uninstall -y torch torchvision torchaudio transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Found existing installation: transformers 5.6.2
Uninstalling transformers-5.6.2:
  Successfully uninstalled transformers-5.6.2
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
ERROR: pip's dependency resolver does not curre

  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.6.2-py3-none-any.whl (10.4 MB)


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import torch.nn as nn
from transformers import BertModel
from torch.optim import AdamW
from sklearn.metrics import mean_absolute_error
import numpy as np


tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 128
BATCH_SIZE = 32

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:

class BertUrgencyRegressor(nn.Module):
    def __init__(self, dropout=0.3, freeze_bert=True):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        self.dropout = nn.Dropout(dropout)
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.pooler_output          # [CLS] token, shape (B, 768)
        cls = self.dropout(cls)
        score = torch.sigmoid(self.regressor(cls))  # (B, 1) in [0,1]
        return score.squeeze(1)                     # (B,)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [5]:
def load_urgency_model(model_path, device=None):
    """Load a saved BertUrgencyRegressor from disk."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = BertUrgencyRegressor(dropout=0.3).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print(f"Model loaded from {model_path} on {device}")
    return model

In [6]:
def predict_urgency(text, model, tokenizer=tokenizer, max_len=128, device=None):
    """
    Takes a raw ticket string, returns urgency as a float in [0, 3].
    Higher = more urgent.
    """
    if device is None:
        device = next(model.parameters()).device

    text = ' '.join(text.strip().split())

    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=max_len,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        score = model(input_ids, attention_mask)

    urgency = 3.0 - (score.item() * 3.0)  # flip: higher = more urgent
    return urgency/3.0

In [7]:
model_save_path_refined = PROJ + 'bert_urgency_refined.pth'
model = load_urgency_model(model_save_path_refined)



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_8898/2023255000.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrar

Model loaded from /content/drive/MyDrive/TAC_459/bert_urgency_refined.pth on cuda


# MODEL IS IN; INFERENCE

In [14]:
samples = [
    "server is completely down cannot access anything urgent",
    "please update my profile picture when you get a chance",
    "cannot connect to vpn blocking all work need immediate help",
    "question about subscription renewal next month",
    "My account is locked and I can't log in. I need access immediately.",
    "I have a quick question about a feature in the new update.",
    "Our entire system is down, impacting all customers globally. Critical emergency!",
    "Can you help me reset my password?",
    "The application is crashing every time I try to open it. It's affecting my work.",
    "I found a typo on the website. No rush to fix it.",
    "Data loss in production environment. All hands on deck!",
    "I'd like to request a new report format for next quarter's meeting.",
    "My laptop isn't turning on. I have a presentation in an hour.",
    "The printer is out of ink, can someone replace it?",
    "I need to set up a new user account for a new employee starting next week.",
    "GLOBAL OUTAGE: All services are down, revenue impact is critical and immediate!",
    "Could you please add a new emoji to our internal chat system next week?",
    "My workstation is completely unresponsive, I cannot do any work, this is extremely critical!",
    "I have a suggestion for improving a non-critical UI element.",
    "A minor bug is causing a display issue for some users, but it's not affecting core functionality.",
    "The entire database is corrupted, we need immediate intervention to prevent total data loss!",
    "I need help changing my desktop background, when you have a moment.",
    "System is intermittently slow, but still functional. Users are complaining.",
    "The cafeteria ran out of coffee, this is a low priority request.",
    "I need to schedule a meeting with a new client next month, please assist."
]

tickets = pd.DataFrame(samples)
tickets['urgency'] = tickets[0].apply(lambda x: predict_urgency(x, model))
tickets.sort_values(by='urgency', ascending=False, inplace=True)
tickets.reset_index(drop=True, inplace=True)
print(tickets)

                                                    0   urgency
0   server is completely down cannot access anythi...  0.953815
1   My account is locked and I can't log in. I nee...  0.767625
2   System is intermittently slow, but still funct...  0.738132
3   cannot connect to vpn blocking all work need i...  0.722444
4   Data loss in production environment. All hands...  0.710106
5   Our entire system is down, impacting all custo...  0.704726
6   GLOBAL OUTAGE: All services are down, revenue ...  0.668674
7   The entire database is corrupted, we need imme...  0.650292
8                  Can you help me reset my password?  0.645954
9   Could you please add a new emoji to our intern...  0.545314
10  A minor bug is causing a display issue for som...  0.539106
11  My workstation is completely unresponsive, I c...  0.522978
12  The printer is out of ink, can someone replace...  0.505599
13  I have a suggestion for improving a non-critic...  0.498497
14  I need help changing my desktop back